In [164]:
import pandas as pd
import glob
import os
from datetime import timedelta
import pendulum
from clickhouse_driver import Client
import pymysql

host72 = "192.168.1.72"
host128 = "192.168.1.182"

Con = pymysql.connect(host=host128, user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

client = Client(host='192.168.1.99', port='9000', user='default', password='jdfwl6812hwe',
                database='suitecrm_robot_ch', settings={'use_numpy': True})

In [166]:
steps_sql = r'C:\Users\Guest\Desktop\sql\25 отчет\steps.sql'
Total_calls_last_week_sql = r'C:\Users\Guest\Desktop\sql\25 отчет\Total_calls_last_week.sql'
Transfer_steps_sql = r'C:\Users\Guest\Desktop\sql\25 отчет\Transfer_steps.sql' 
transfers_sql = r'C:\Users\Guest\Desktop\sql\25 отчет\transfers.sql' 

In [ ]:
Con = pymysql.connect(host=host128, user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')
steps = pd.read_sql_query(open(steps_sql, 'r').read(), Con).fillna('')
print('steps_sql Done')

Con = pymysql.connect(host=host128, user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')

Transfer_steps = pd.read_sql_query(open(Transfer_steps_sql, 'r').read(), Con).fillna('')
print('CallWaitUser Done')
transfers = pd.read_sql_query(open(transfers_sql, 'r').read(), Con).fillna('')
print('CallWaitUser Done')

Con = pymysql.connect(host=host128, user="base_dep_slave", passwd="IyHBh9mDBdpg", db="suitecrm",
                      charset='utf8')
Total_calls_last_week = pd.read_sql_query(open(Total_calls_last_week_sql, 'r').read(), Con).fillna('')
print('Total_calls_last_week_sql Done')

In [170]:
def queue_project2():
    path = '//192.168.1.157/dbs/scripts fsp/Current Files/Проект/Очереди'
    files = sorted(glob.glob(path + "/*.csv"), reverse=True)
    project_queue = pd.DataFrame()
    n = 0
    num_of_files = len(os.listdir(path))

    print(f'Всего файлов {num_of_files}')

    for i in files:
        n += 1
        df = pd.read_csv(i)
        project_queue = pd.concat([project_queue, df], ignore_index=True, axis=0)
        # project_queue = project_queue.append(df)
    del df

    # print(project_queue.columns)

    project_queue = project_queue.rename(columns={'Группировка': 'type_ro'})
    project_queue['date'] = project_queue['date'].astype('str')

    project_queue['RN'] = project_queue.groupby(['Очередь', 'date']).cumcount() + 1
    project_queue = project_queue[project_queue['RN'] == 1][['Очередь', 'type_ro', 'date']]

    return project_queue

def perevod(row):
    if row['step'] == '':
        return 0
    else:
        return 1
    
def perevelys(row):
    if row['step'] == '':
        return 0
    elif row['assigned_user_id'] in ['1','',' ','0']:
        return 0
    else:
        return 1
    
def etv(row):
    for i in row['have_ptv'].split(','):
        if i in row['route'].split(','):
            return i
        else:
            return '0'
        
def last_step(row):
    for i in row['steps_inconvenient'].split(','):
        if i == row['last_step']:
            return 'steps_inconvenient'
        
    for i in row['steps_error'].split(','):
        if i == row['last_step']:
            return 'steps_error'
        
    for i in row['steps_refusing'].split(','):
        if i == row['last_step']:
            return 'steps_refusing'
        
    for i in row['top_recall'].split(','):
        if i == row['last_step']:
            return 'top_recall'
        
    for i in row['hello_end'].split(','):
        if i == row['last_step']:
            return 'hello_end'
        
    for i in row['welcome_end'].split(','):
        if i == row['last_step']:
            return 'welcome_end'
        
    for i in row['ntv'].split(','):
        if i == row['last_step']:
            return 'ntv'
        
    for i in row['abonent'].split(','):
        if i == row['last_step']:
            return 'abonent'
        
        
def network_provider_c(i):
    if i in {'10','68'}:
        return 'Теле 2'
    elif i == '80':
        return 'Билайн'
    elif i == '82' or  i == '63':
        return 'Мегафон'
    elif i == '83':
        return 'МТС'
    else:
        return 'MVNO'
    
ptv_nasha = ['^5^', '^6^', '^3^', '^10^', '^11^', '^19^', ]
ptv_ne_nasha = ['^5_15^', '^5_16^', '^5_17^', '5_18^', '^5_19^', '^5_20^', '^5_21^',
                '^6_15^', '^6_16^', '^6_17^', '6_18^', '^6_19^', '^6_20^', '^6_21^',
                '^3_15^', '^3_16^', '^3_17^', '3_18^', '^3_19^', '^3_20^', '^3_21^',
                '^10_15^', '^10_16^', '^10_17^', '10_18^', '^10_19^', '^10_20^', '^10_21^',
                '^11_15^', '^11_16^', '^11_17^', '11_18^', '^11_19^', '^11_20^', '^11_21^',
                '^19_15^', '^19_16^', '^19_17^', '19_18^', '^19_19^', '^19_20^', '^19_21^']

def region(row):
    if any(w in row['ptv_c'] for w in ptv_nasha):
        return 'ptv_1'
    elif any(w in row['ptv_c'] for w in ptv_ne_nasha):
        return 'ptv_2'
    else:
        return row['region_c']
    


def network_provider_c(i):
    if i in {'10','68'}:
        return 'Теле 2'
    elif i == '80':
        return 'Билайн'
    elif i == '82' or i == '63':
        return 'Мегафон'
    elif i == '83':
        return 'МТС'
    else:
        return 'MVNO'
    
ptv_nasha = ['^5^', '^6^', '^3^', '^10^', '^11^', '^19^', ]
ptv_ne_nasha = ['^5_15^', '^5_16^', '^5_17^', '5_18^', '^5_19^', '^5_20^', '^5_21^',
                '^6_15^', '^6_16^', '^6_17^', '6_18^', '^6_19^', '^6_20^', '^6_21^',
                '^3_15^', '^3_16^', '^3_17^', '3_18^', '^3_19^', '^3_20^', '^3_21^',
                '^10_15^', '^10_16^', '^10_17^', '10_18^', '^10_19^', '^10_20^', '^10_21^',
                '^11_15^', '^11_16^', '^11_17^', '11_18^', '^11_19^', '^11_20^', '^11_21^',
                '^19_15^', '^19_16^', '^19_17^', '19_18^', '^19_19^', '^19_20^', '^19_21^']

def region(row):
    if any(w in row['ptv_c'] for w in ptv_nasha):
        return 'ptv_1'
    elif any(w in row['ptv_c'] for w in ptv_ne_nasha):
        return 'ptv_2'
    else:
        return row['region_c']
    
    
print('-- качества')
calls['network_provider_c'] = calls['network_provider_c'].astype('str').apply(lambda x: network_provider_c(x))
calls['region'] = calls.apply(lambda row: region(row), axis=1)
        # calls['region'] = ''

print('Крепим Тип РО')
calls = calls.merge(queue_project, how='left',left_on=['queue','call_date'],right_on=['queue','date'])

print('Определяем шаги')
calls['description'] = calls.apply(lambda row: last_step(row), axis=1)
calls['etv'] = calls.apply(lambda row: etv(row), axis=1) 

region = pd.read_csv(r'C:\Users\Guest\Desktop\sql\25 отчет\region.csv',  sep=',', encoding='utf-8').fillna('')
region['region'] = region['region'].astype('str')

print('Группируем')
calls = calls.groupby(['call_date',
       'call_hour',
       'call_minute',
       'queue',
       'destination_queue',
       'directory',
       'assigned_user_id',
       'last_step',
       'description',
       'count_steps',
       'client_status',
       'otkaz',
       'inbound_call',
       'region',
       'marker',  
       'network_provider_c',
       'city_c',
       'town',
       'etv',
       'type_ro',
       'was_stepgroups'],as_index=False, dropna=False).agg({'contact_id_c': 'count',
                                                 'was_repeat': 'sum',
                                                 'perevod': 'sum',
                                                 'perevelys': 'sum',
                                                 'billsec': 'sum'}).rename(columns={'contact_id_c': 'calls',
                                                                                    'was_repeat': 'was_ptv'})


print('Заменяем названия справочниками')
calls[['city_c','town','region']] = calls[['city_c','town','region']].astype('str')
calls = calls.merge(city, how='left', on='city_c')
calls = calls.merge(town, how='left', on='town')
calls = calls.merge(region, how='left', on='region')


calls = calls[['call_date',
       'call_hour',
       'call_minute',
       'queue',
       'destination_queue',
       'directory',
       'assigned_user_id',
       'last_step',
       'description',
       'count_steps',
       'client_status',
       'otkaz',
       'inbound_call',
       'region',
       'marker',  
       'network_provider_c',
       'city_c',
       'town',
       'etv',
       'was_stepgroups',
       'city_name',
       'town_name',
       'region_name',
       'type_ro',
       'calls',
       'was_ptv',
       'perevod',
       'perevelys',
        'billsec']]


calls['queue'] = calls['queue'].astype('int64')
calls['destination_queue'] = pd.to_numeric(calls['destination_queue'], errors='coerce')
calls['destination_queue'] = calls['destination_queue'].fillna(0)
calls['destination_queue'] = calls['destination_queue'].astype('int64')
calls['last_step'] = calls['last_step'].astype('int64')
calls['etv'] = pd.to_numeric(calls['etv'], errors='coerce')
calls['etv'] = calls['etv'].fillna(0)
calls['etv'] = calls['etv'].astype('int64')


calls = calls[calls['queue'] != '50-n']
print('Редактируем формат')
calls['call_date'] = pd.to_datetime(calls['call_date'])
calls[['etv','calls','was_ptv','perevod','perevelys','billsec','call_hour','call_minute',
'count_steps','last_step','destination_queue','queue']] = calls[['etv','calls','was_ptv','perevod','perevelys','billsec','call_hour','call_minute',
'count_steps','last_step','destination_queue','queue']].fillna(0).astype('int64')
calls[['directory','assigned_user_id','client_status','otkaz','inbound_call','was_stepgroups','type_ro',
'network_provider_c','marker','region_name','town_name','city_name']] = calls[['directory','assigned_user_id','client_status','otkaz','inbound_call','was_stepgroups','type_ro',
'network_provider_c','marker','region_name','town_name','city_name']].fillna('').astype('str')

-- дата
Шаги переводов
Переводы
(1358621, 5)
Описание шагов
Справочники
Всего файлов 241
  queue type_ro        date
0  9001  Другие  2024-03-28
1  9002  Другие  2024-03-28
2  9003  Другие  2024-03-28
Соединяем
Редактируем
-- переводы
-- качества
Крепим Тип РО
Определяем шаги
Группируем
Заменяем названия справочниками
Редактируем формат


In [ ]:
calls = Total_calls_last_week.fillna('')
calls['last_step'] = calls['last_step'].astype('str').apply(lambda x: x.replace('.0',''))
calls['queue'] = calls['queue'].astype('str').apply(lambda x: x.replace('.0',''))
calls['phone'] = calls['phone'].astype('str').apply(lambda x: x.replace('.0',''))
calls['call_date'] = pd.to_datetime(calls['call_date'])
print('-- дата')
calls['call_hour'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.hour + 3)
calls['call_minute'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.minute)
calls['call_date'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.date())

print('Шаги переводов')
transfer_steps = Transfer_steps.fillna('')
transfer_steps['step'] = transfer_steps['step'].astype('str').apply(lambda x: x.replace('.0',''))
transfer_steps['ochered'] = transfer_steps['ochered'].astype('str').apply(lambda x: x.replace('.0',''))

print('Переводы')
transfers =transfers.fillna('')
transfers['phone'] = transfers['phone'].astype('str').apply(lambda x: x.replace('.0',''))
transfers['dialog'] = transfers['dialog'].astype('str').apply(lambda x: x.replace('.0',''))
transfers['date'] = pd.to_datetime(transfers['date']).apply(lambda x: x.date())
print(transfers.shape)

print('Описание шагов')
steps = steps.fillna('')
steps['queue'] = steps['queue'].astype('str')

print('Справочники')
city = pd.read_csv(r'C:\Users\Guest\Desktop\sql\25 отчет\city.csv',  sep=',', encoding='utf-8').fillna('')
city['city_c'] = city['city_c'].astype('str')
town = pd.read_csv(r'C:\Users\Guest\Desktop\sql\25 отчет\town.csv',  sep=',', encoding='utf-8').fillna('')
town['town'] = town['town'].astype('str')
region = pd.read_csv(r'C:\Users\Guest\Desktop\sql\25 отчет\region.csv',  sep=',', encoding='utf-8').fillna('')
region['region'] = region['region'].astype('str')

queue_project = queue_project2().fillna(0)
queue_project['date'] = pd.to_datetime(queue_project['date']).apply(lambda x: x.date())
queue_project = queue_project.rename(columns={'Очередь': 'queue'})
queue_project['queue'] = queue_project['queue'].astype('int').astype('str')
print(queue_project.head(3))

print('Соединяем')
calls = calls.merge(transfer_steps, left_on = ['last_step','queue'], right_on = ['step','ochered'], how = 'left')
calls = calls.merge(steps, how='left',on='queue')
calls = calls.merge(transfers,how='left', left_on = ['phone','queue','call_date'], right_on = ['phone','dialog','date'])
calls.fillna('', inplace=True)

print('Редактируем')
# print('-- дата')
        # calls['call_hour'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.hour + 3)
        # calls['call_minute'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.minute)
        # calls['call_date'] = pd.to_datetime(calls['call_date']).apply(lambda x: x.date())
print('-- переводы')
calls['perevod'] = calls.apply(lambda row: perevod(row), axis=1)
calls['perevelys'] = calls.apply(lambda row: perevelys(row), axis=1)

In [172]:
client = Client(host='192.168.1.99', port='9000', user='default', password='jdfwl6812hwe',
                database='suitecrm_robot_ch', settings={'use_numpy': True})
client.insert_dataframe('INSERT INTO suitecrm_robot_ch.report_25_archive VALUES', calls)  

6496777